# GPT-4o-mini zero-shot utterance generation → downstream emotion

This notebook performs the missing OpenAI run end to end:

1. Extract the uploaded GitHub ZIP.
2. Generate one zero-shot GPT-4o-mini future utterance for every canonical IEMOCAP test point.
3. Label every generated utterance using the same fixed Qwen2.5-7B-Instruct emotion labeler used for all other sources.
4. Save resumable outputs under `/workspace/utterance/openai_gpt4omini/`.

Set `OPENAI_API_KEY` in the pod environment before running.

In [ ]:
from pathlib import Path
import os, shutil, zipfile

SOURCE_ID = "openai_gpt4omini"
SOURCE_TITLE = "GPT-4o-mini zero-shot predicted utterances"
OPENAI_MODEL = "gpt-4o-mini"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
WORKSPACE = Path("/workspace")
EXTRACT_DIR = WORKSPACE / "github_repo_extracted"
OUTPUT_DIR = str(WORKSPACE / "utterance" / SOURCE_ID)
MAX_SAMPLES = 0
BATCH_SIZE = 8
CHECKPOINT_EVERY = 40
SEED = 42

# Upload the GitHub ZIP for provenance/consistency and the IEMOCAP pickle.
zip_candidates = sorted(WORKSPACE.glob("*.zip"))
repo_zips = []
for candidate in zip_candidates:
    try:
        with zipfile.ZipFile(candidate) as zf:
            names = zf.namelist()
            if any(name.endswith("zeroshot_utterance/openai_api/openai_nextutt.py") for name in names):
                repo_zips.append(candidate)
    except zipfile.BadZipFile:
        pass
if len(repo_zips) != 1:
    raise RuntimeError(f"Expected exactly one GitHub ZIP containing openai_nextutt.py, found: {repo_zips}")
REPO_ZIP = repo_zips[0]
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(REPO_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

data_matches = list(WORKSPACE.glob("IEMOCAP_features.pkl")) + list(EXTRACT_DIR.rglob("IEMOCAP_features.pkl"))
data_matches = list(dict.fromkeys(str(p.resolve()) for p in data_matches))
if len(data_matches) != 1:
    raise RuntimeError(f"Expected exactly one IEMOCAP_features.pkl, found: {data_matches}")
DATA_PATH = data_matches[0]
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the pod environment before running this cell.")
print("Repository ZIP:", REPO_ZIP)
print("Dataset:", DATA_PATH)
print("Output:", OUTPUT_DIR)
print("OpenAI model:", OPENAI_MODEL)

## One-time environment setup

In [ ]:
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--upgrade",
    "transformers==4.46.3", "tokenizers==0.20.3", "huggingface-hub==0.26.2",
    "safetensors==0.4.5", "accelerate==1.1.1",
])
print("Installed compatible Hugging Face packages without replacing Torch/CUDA.")
print("Restart the kernel once, then rerun Configuration and continue.")

Restart the kernel once, then rerun Configuration and continue.

In [ ]:
import json, pickle, random, re, shutil
from dataclasses import dataclass
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

EMOTION_LABELS = ["neutral", "frustration", "sadness", "anger", "excited", "happiness"]
LABEL2ID = {x: i for i, x in enumerate(EMOTION_LABELS)}
NORMALIZE = {
    "neutral":"neutral", "neu":"neutral",
    "frustration":"frustration", "frustrated":"frustration", "fru":"frustration",
    "sadness":"sadness", "sad":"sadness",
    "anger":"anger", "angry":"anger", "ang":"anger",
    "excited":"excited", "excitement":"excited", "exc":"excited",
    "happiness":"happiness", "happy":"happiness", "hap":"happiness",
}

@dataclass
class Sample:
    dialogue_id: str
    history: list
    history_speakers: list
    history_emotions: list
    target_speaker: str
    target_emotion: str
    target_emotion_id: int

def normalize_label(x):
    if isinstance(x, int):
        return EMOTION_LABELS[x]
    s = str(x).strip().lower()
    if s.isdigit() and int(s) < len(EMOTION_LABELS):
        return EMOTION_LABELS[int(s)]
    return NORMALIZE.get(s, s)

def speaker_label(x):
    try:
        import numpy as np
        if isinstance(x, (list, tuple, np.ndarray)):
            return "SpeakerA" if int(np.argmax(x)) == 0 else "SpeakerB"
    except Exception:
        pass
    return str(x)

def carve_val(train_vids, n_dev=20):
    ids = sorted(train_vids)
    return set(ids[:-n_dev]), set(ids[-n_dev:])

def emit_samples(vid, utterances, speakers, labels):
    speakers = [speaker_label(x) for x in speakers]
    labels = [normalize_label(x) for x in labels]
    rows = []
    for t in range(1, len(utterances)):
        rows.append(Sample(
            dialogue_id=f"{vid}_t{t}",
            history=list(utterances[:t]),
            history_speakers=list(speakers[:t]),
            history_emotions=list(labels[:t]),
            target_speaker=speakers[t],
            target_emotion=labels[t],
            target_emotion_id=LABEL2ID[labels[t]],
        ))
    return rows

def load_canonical_iemocap(path):
    with open(path, "rb") as f:
        raw = pickle.load(f, encoding="latin1")
    if isinstance(raw, (list, tuple)) and len(raw) >= 9:
        video_speakers, video_labels, video_sentence = raw[1], raw[2], raw[6]
        train_vids, test_vids = list(raw[7]), set(raw[8])
    elif isinstance(raw, dict) and ("videoSentence" in raw or "trainVid" in raw):
        video_speakers = raw["videoSpeakers"]
        video_labels = raw["videoLabels"]
        video_sentence = raw["videoSentence"]
        train_vids, test_vids = list(raw["trainVid"]), set(raw["testVid"])
    else:
        raise ValueError("Expected the standard DialogueRNN-style IEMOCAP pickle.")
    train_set, dev_set = carve_val(train_vids, 20)
    splits = {"train": [], "dev": [], "test": []}
    for vid, utts in video_sentence.items():
        if vid in test_vids:
            split = "test"
        elif vid in dev_set:
            split = "dev"
        elif vid in train_set:
            split = "train"
        else:
            continue
        splits[split].extend(emit_samples(vid, utts, video_speakers[vid], video_labels[vid]))
    return splits

def parse_dialogue_id(dialogue_id):
    m = re.match(r"^(.*)_t(\d+)$", dialogue_id)
    if not m:
        raise ValueError(f"Bad dialogue_id: {dialogue_id}")
    return m.group(1), int(m.group(2))

def load_predicted_utterances(path):
    lookup = {}
    with open(path, encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not all(k in row for k in ("vid", "t", "pred")):
                raise ValueError(f"Line {ln} is missing vid/t/pred: {row.keys()}")
            lookup[(str(row["vid"]), int(row["t"]))] = str(row.get("pred") or "")
    return lookup

def format_history(s, max_turns=10):
    return "\n".join(
        f"Speaker {spk} ({emo}): {utt}"
        for spk, emo, utt in zip(
            s.history_speakers[-max_turns:],
            s.history_emotions[-max_turns:],
            s.history[-max_turns:],
        )
    )

def same_speaker_shift(s):
    for spk, emo in zip(reversed(s.history_speakers), reversed(s.history_emotions)):
        if spk == s.target_speaker:
            return s.target_emotion != emo
    return None

def parse_emotion(text):
    low = str(text).lower()
    tagged = re.search(r"<emotion>\s*([^<\n]+)", low)
    if tagged:
        token = tagged.group(1).strip().split()[0].strip(".,:;!?")
        if token in NORMALIZE:
            return NORMALIZE[token]
    hits = []
    for label in EMOTION_LABELS:
        for match in re.finditer(rf"\b{re.escape(label)}\b", low):
            hits.append((match.start(), label))
    return max(hits)[1] if hits else None

def compute_metrics(records, samples):
    by_id = {r["dialogue_id"]: r for r in records}
    if len(by_id) != len(samples):
        raise RuntimeError(f"Expected {len(samples)} records, found {len(by_id)}")
    y_true = [s.target_emotion_id for s in samples]
    y_pred = [LABEL2ID.get(by_id[s.dialogue_id].get("predicted_emotion"), -1) for s in samples]

    def score(indices):
        if not indices:
            return {"n": 0, "weighted_f1": 0.0, "macro_f1": 0.0, "accuracy": 0.0}
        yt = [y_true[i] for i in indices]
        yp = [y_pred[i] for i in indices]
        per_class = []
        for lab in range(6):
            tp = sum(a == lab and b == lab for a, b in zip(yt, yp))
            fp = sum(a != lab and b == lab for a, b in zip(yt, yp))
            fn = sum(a == lab and b != lab for a, b in zip(yt, yp))
            p = tp / (tp + fp) if tp + fp else 0.0
            r = tp / (tp + fn) if tp + fn else 0.0
            f1 = 2 * p * r / (p + r) if p + r else 0.0
            support = sum(a == lab for a in yt)
            per_class.append((f1, support))
        n = len(indices)
        return {
            "n": n,
            "weighted_f1": sum(f * sup for f, sup in per_class) / n,
            "macro_f1": sum(f for f, _ in per_class) / 6,
            "accuracy": sum(a == b for a, b in zip(yt, yp)) / n,
        }

    all_idx = list(range(len(samples)))
    es_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is True]
    ns_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is False]
    overall = score(all_idx)
    es = score(es_idx)
    ns = score(ns_idx)
    return {
        "n": len(samples),
        "weighted_f1": overall["weighted_f1"],
        "macro_f1": overall["macro_f1"],
        "accuracy": overall["accuracy"],
        "parse_failures": sum(x == -1 for x in y_pred),
        "es_n": es["n"],
        "es_weighted_f1": es["weighted_f1"],
        "no_shift_n": ns["n"],
        "no_shift_weighted_f1": ns["weighted_f1"],
        "undefined_n": len(samples) - len(es_idx) - len(ns_idx),
    }


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "CUDA GPU not visible."
splits = load_canonical_iemocap(DATA_PATH)
all_samples = splits["test"]
assert len(all_samples) == 1592, len(all_samples)
samples = all_samples if MAX_SAMPLES == 0 else all_samples[:MAX_SAMPLES]
print({k: len(v) for k, v in splits.items()})
print("Run size:", len(samples))

## Generate GPT-4o-mini utterances (resumable)

In [ ]:
import os, time, requests

GEN_SYSTEM = (
    "You continue conversations. Given the dialogue so far, write ONLY the next line "
    "that the named speaker would say. Output just the utterance text, nothing else, "
    "no speaker name, no quotes, no explanation."
)

def build_generation_prompt(s, max_turns=12):
    lines = [
        f"{spk} ({emo}): {utt}"
        for spk, emo, utt in zip(
            s.history_speakers[-max_turns:],
            s.history_emotions[-max_turns:],
            s.history[-max_turns:],
        )
    ]
    return "\n".join(lines) + f"\n{s.target_speaker}:"

def clean_generated_utterance(text, next_speaker):
    out = str(text or "").strip()
    out = re.sub(rf"^{re.escape(next_speaker)}\s*:\s*", "", out, flags=re.I)
    out = re.sub(r"^(sure|here'?s?|the next line( is)?|response)\s*[:,-]?\s*", "", out, flags=re.I)
    return out.strip().strip('"').split("\n")[0].strip()

def call_openai(prompt, attempts=6):
    headers = {
        "Authorization": "Bearer " + os.environ["OPENAI_API_KEY"],
        "Content-Type": "application/json",
    }
    payload = {
        "model": OPENAI_MODEL,
        "messages": [
            {"role": "system", "content": GEN_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
        "max_tokens": 64,
    }
    for attempt in range(attempts):
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120,
        )
        if response.status_code == 200:
            data = response.json()
            return data["choices"][0]["message"]["content"] or ""
        if response.status_code in (429, 500, 502, 503, 504):
            wait = min(60, 2 ** attempt)
            print(f"OpenAI retry {attempt + 1}/{attempts} after HTTP {response.status_code}; sleeping {wait}s")
            time.sleep(wait)
            continue
        raise RuntimeError(f"OpenAI API error {response.status_code}: {response.text[:1000]}")
    raise RuntimeError("OpenAI API failed after retries.")


In [ ]:
# Mandatory API smoke test: one request only.
s = samples[0]
raw = call_openai(build_generation_prompt(s))
pred = clean_generated_utterance(raw, s.target_speaker)
print("Raw:", raw)
print("Cleaned:", pred)
if not pred:
    raise RuntimeError("OpenAI smoke test returned an empty utterance.")
print("OPENAI SMOKE TEST PASSED")

## Load fixed Qwen2.5-7B emotion labeler

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.is_available(), "A CUDA GPU is required."
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model.eval()
print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Dtype:", compute_dtype)
print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


In [ ]:
SYSTEM_PROMPT = """You are an expert in conversational emotion analysis.
You are given a dialogue history with speaker and emotion labels, followed by a PREDICTED next utterance for the known next speaker. Classify the emotion expressed by that predicted utterance while using the dialogue history as context.

Valid emotions: neutral, frustration, sadness, anger, excited, happiness

Reply exactly as:
<emotion>one_label</emotion>"""

def make_label_prompt(s, predicted_utterance):
    return (
        SYSTEM_PROMPT
        + "\n\nDialogue history:\n"
        + format_history(s)
        + f"\n\nSpeaker {s.target_speaker} is predicted to say:\n{predicted_utterance}"
        + "\n\nEmotion:"
    )

@torch.inference_mode()
def generate_labels(prompts, max_new_tokens=24):
    rendered = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for p in prompts
    ]
    toks = tokenizer(
        rendered,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    )
    toks = {k: v.to(model.device) for k, v in toks.items()}
    out = model.generate(
        **toks,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    input_width = toks["input_ids"].shape[1]
    return [tokenizer.decode(row[input_width:], skip_special_tokens=True) for row in out]


## Generate all utterances, then label them (both stages resume)

In [ ]:
utterance_path = Path(OUTPUT_DIR) / "openai_predicted_utterances.jsonl"
existing_utterances = {}
if utterance_path.exists():
    with utterance_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                existing_utterances[(str(row["vid"]), int(row["t"]))] = row
pending_generation = [s for s in samples if parse_dialogue_id(s.dialogue_id) not in existing_utterances]
print("Generated already:", len(existing_utterances), "Pending:", len(pending_generation))

with utterance_path.open("a", encoding="utf-8") as fout:
    for i, s in enumerate(pending_generation, 1):
        raw = call_openai(build_generation_prompt(s))
        pred = clean_generated_utterance(raw, s.target_speaker)
        if not pred:
            raise RuntimeError(f"Empty OpenAI utterance for {s.dialogue_id}")
        vid, turn = parse_dialogue_id(s.dialogue_id)
        row = {"vid": vid, "t": turn, "pred": pred, "ref": None, "raw": raw}
        fout.write(json.dumps(row, ensure_ascii=False) + "\n")
        fout.flush()
        if i % 25 == 0 or i == len(pending_generation):
            print(f"Generated {len(existing_utterances) + i}/{len(samples)}", flush=True)

pred_lookup = load_predicted_utterances(utterance_path)
if len(pred_lookup) != len(samples):
    raise RuntimeError(f"Expected {len(samples)} generated utterances, found {len(pred_lookup)}")
print("OpenAI utterance generation complete.")

checkpoint_path = Path(OUTPUT_DIR) / "predictions_with_emotions.jsonl"
existing = {}
if checkpoint_path.exists():
    with checkpoint_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                existing[row["dialogue_id"]] = row
pending = [s for s in samples if s.dialogue_id not in existing]
print("Already complete:", len(existing), "Pending:", len(pending))

with checkpoint_path.open("a", encoding="utf-8") as fout:
    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending[start:start + BATCH_SIZE]
        utterances = [pred_lookup[parse_dialogue_id(s.dialogue_id)] for s in batch]
        prompts = [make_label_prompt(s, utt) for s, utt in zip(batch, utterances)]
        raw_outputs = generate_labels(prompts)
        for s, utt, raw in zip(batch, utterances, raw_outputs):
            vid, turn = parse_dialogue_id(s.dialogue_id)
            rec = {
                "dialogue_id": s.dialogue_id,
                "vid": vid,
                "t": turn,
                "source_id": SOURCE_ID,
                "predicted_utterance": utt,
                "gold_emotion": s.target_emotion,
                "predicted_emotion": parse_emotion(raw),
                "raw_labeler_output": raw,
                "is_emotion_shift": same_speaker_shift(s),
            }
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()
        done = len(existing) + min(start + len(batch), len(pending))
        if done % CHECKPOINT_EVERY < BATCH_SIZE or done == len(samples):
            print(f"{done}/{len(samples)}", flush=True)
print("Full run complete")


## Evaluate and package

In [ ]:
records = []
with checkpoint_path.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))
metrics = compute_metrics(records, samples)
metrics.update({
    "source_id": SOURCE_ID,
    "source_title": SOURCE_TITLE,
    "labeler_model": MODEL_ID,
})
metrics_path = Path(OUTPUT_DIR) / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(Path(OUTPUT_DIR) / "README.txt").write_text(
    SOURCE_TITLE + "\nFixed downstream labeler: " + MODEL_ID + "\n\n" + json.dumps(metrics, indent=2),
    encoding="utf-8",
)
zip_path = shutil.make_archive(OUTPUT_DIR, "zip", root_dir=OUTPUT_DIR)
print(json.dumps(metrics, indent=2))
print("Saved folder:", OUTPUT_DIR)
print("ZIP:", zip_path)
